[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Artefactos en *Github Actions*.

Un artefacto es un archivo, en formato binarios por lo general, que es el resultado del proceso de compilación o empaquetado durante el ciclo de desarrollo de software.

Al ser una herramineta de *CI/CD*, es de esperarse que en la mayoría de los casos, el resultado de un *workflow* de *Github Actions* sea un artefacto.

Debido a que los *runners* son máquinas efímeras, es necesario extraer y conservar de algún modo los artefactos resultantes de un *workflow*. *Github* actions permite extraer, consumir y guardar artefactos durante el periodo de retención predeterminado, configurable con ```retention-days```.

https://docs.github.com/en/actions/using-workflows/storing-workflow-data-as-artifacts

## Las *actions* de gestión de artefactos.

Las *actions* que permiten realizar operaciones con artefactos son:

* [```actions/upload-artifact```](https://github.com/actions/upload-artifact), la cual permite cargar un archivo o un directorio completo al *runner*.
* [```actions/download-artifact```](https://github.com/actions/download-artifact), la cual permite extraer un archivo o un documento y hacerlos disponible para su descarga.
* De forma interna, [```@actions/artifact```](https://github.com/actions/toolkit/tree/main/packages/artifact) permite gestionar los artefactos.

**Ejemplo:**

El directorio ```api/``` de este repositorio contiene una interfaz de programación de aplicaciones (*API*) escrita en *Python* y *Flask*. El *workflow* ```.github/workflows/ci.yml``` instala las dependencias de desarrollo, ejecuta el linter *ruff* y las pruebas con *pytest*, y publica como artefacto el informe de cobertura ```coverage.xml```.

```yaml
name: CI

on:
  push:
  pull_request:

permissions:
  contents: read

jobs:
  verificar:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: api
    steps:
    - name: Checkout
      uses: actions/checkout@v7

    - name: Configurar Python
      uses: actions/setup-python@v7
      with:
        python-version: "3.12"
        cache: pip
        cache-dependency-path: api/requirements-dev.txt

    - name: Instalar dependencias
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements-dev.txt

    - name: Linter (ruff)
      run: ruff check .

    - name: Pruebas (pytest)
      run: pytest

    - name: Publicar informe de cobertura
      if: always()
      uses: actions/upload-artifact@v7
      with:
        name: cobertura-pytest
        path: api/coverage.xml
        if-no-files-found: ignore

    - name: Construir imagen (sin publicar)
      working-directory: .
      run: docker build -t cd441-api:ci .
```


* El paso ```Pruebas (pytest)``` del *job* ```verificar``` ejecuta las pruebas automatizadas de la *API*. *pytest* deja el informe de cobertura en ```api/coverage.xml```.
* El paso ```Publicar informe de cobertura``` conserva ese archivo como artefacto llamado ```cobertura-pytest```, disponible para su descarga al terminar la ejecución (aunque las pruebas hayan fallado, gracias a ```if: always()```).
* El paso final construye la imagen de contenedor para detectar un ```Dockerfile``` roto, pero **no** la publica en un registro. La publicación se trata en el cuaderno 16.

El cuaderno 14 retoma este *workflow* y lo explica con detalle.


Al finalizar la ejecución del *workflow*, es posible acceder y descargar el contenido de ```cobertura-pytest``` en el apartado *Artifacts*, localizado en el resumen de la ejecución.

![image-2.png](img/11/11-01.png)


![image-2.png](img/11/11-02.png)

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>